# 01 · Extracción de flujos desde los PCAP crudos de Edge-IIoTset

**Investigación:** Aprendizaje federado para mejorar la capacidad de detección de los sistemas de detección de intrusiones en redes IoT de borde, 2026.

Este notebook convierte las capturas **PCAP crudas** de Edge-IIoTset (Ferrag et al., 2022) en una tabla de **flujos bidireccionales etiquetados**. No se usan los CSV preprocesados del dataset (`DNN-EdgeIIoT` / `ML-EdgeIIoT`): todo el preprocesamiento se hace aquí, desde los paquetes.

**Entrada:** carpeta del dataset descomprimida en `data/raw/` (se busca de forma recursiva).
**Salida:** `data/processed/flows_labeled.parquet` + tablas de diagnóstico en `results/`.

In [1]:
import os, sys, platform
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from fl_ids.flows import FEATURES
from fl_ids.dataset import discover_pcaps, extract_all, label_attack_flows, build_dataset

# ------------------------- CONFIGURACIÓN -------------------------
RAW_DIR   = Path(os.environ.get("EDGE_RAW_DIR", ROOT / "data" / "raw"))   # dataset descomprimido
CACHE_DIR = ROOT / "data" / "interim" / "flows"                          # flujos por archivo (caché)
OUT_FILE  = ROOT / "data" / "processed" / "flows_labeled.parquet"
RES_DIR   = ROOT / "results"; RES_DIR.mkdir(exist_ok=True)

MAX_PACKETS_PER_PCAP = 1_000_000   # límite por captura (las inundaciones DDoS son enormes)
IDLE_TIMEOUT, ACTIVE_TIMEOUT = 60, 120     # segundos: inactividad / ventana máxima por flujo
LABEL_MODE   = "ip_no_vista"       # "ip_no_vista" | "manual" | "archivo"  (ver Paso 3)
ATTACKER_IPS = None                # solo si LABEL_MODE = "manual", p. ej. {"192.168.0.170"}
MAX_PER_CLASS = 15_000             # flujos máximos por clase / por sensor
print("Python", platform.python_version(), "| datos en:", RAW_DIR)

Python 3.13.16 | datos en: /mnt/user-data/uploads/Downloads/fl-ids-edgeiiotset/data/raw


## Paso 1 · Inventario de capturas
Se clasifican los archivos en **normal** (carpeta `Normal traffic/<Sensor>/`) y **ataque** (`*_attack.pcap`). Revisa que aparezcan los 10 sensores y los 14 ataques.

In [2]:
inv = discover_pcaps(RAW_DIR)
assert len(inv), f"No se encontraron .pcap en {RAW_DIR}. Descomprime el dataset ahí (ver README)."
print(inv.groupby("kind")["size_MB"].agg(["count", "sum"]))
inv[["kind", "name", "size_MB"]]

        count     sum
kind                 
attack     14  1133.2
normal     10  1072.6


,kind,name,size_MB
0,attack,Backdoor,6.7
1,attack,DDoS_HTTP_Flood,28.3
2,attack,DDoS_ICMP_Flood,212.4
3,attack,DDoS_TCP_SYN_Flood,282.6
4,attack,DDoS_UDP_Flood,247.6
5,attack,MITM,0.1
6,attack,OS_Fingerprinting,0.1
7,attack,Password,194.6
8,attack,Port_Scanning,1.8
9,attack,Ransomware,2.8


## Paso 2 · Extracción de flujos bidireccionales
Cada flujo agrupa los paquetes de una misma 5-tupla (protocolo, IP y puerto de ambos extremos) en ambos sentidos y termina por inactividad (60 s), al cumplir una ventana máxima de 120 s (así las conexiones MQTT/Modbus persistentes, que duran horas, se dividen en varias observaciones) o al final de la captura. Por cada flujo se calculan **31 características estadísticas** (tamaños, tiempos entre llegadas, banderas TCP, tasas). Las IP, puertos y marcas de tiempo se guardan **solo como metadatos** y no entran al modelo, para que aprenda el *comportamiento* del tráfico y no la identidad de los equipos.

Los resultados se guardan en caché por archivo: si se interrumpe, al volver a ejecutar continúa donde se quedó. Tiempo estimado: 1–2 min por cada millón de paquetes.

> Si cambias `IDLE_TIMEOUT`, `ACTIVE_TIMEOUT` o `MAX_PACKETS_PER_PCAP`, borra la carpeta `data/interim/flows` para que se vuelva a extraer.

In [3]:
flows = extract_all(inv, CACHE_DIR, max_packets=MAX_PACKETS_PER_PCAP,
                    idle_timeout=IDLE_TIMEOUT, active_timeout=ACTIVE_TIMEOUT)
print("Características:", len(FEATURES))

[1/24] Backdoor: 1,471 flujos (desde caché)
[2/24] DDoS_HTTP_Flood: 12,368 flujos (desde caché)


[3/24] DDoS_ICMP_Flood: 809,594 flujos (desde caché)


[4/24] DDoS_TCP_SYN_Flood: 572,947 flujos (desde caché)


[5/24] DDoS_UDP_Flood: 999,929 flujos (desde caché)
[6/24] MITM: 244 flujos (desde caché)
[7/24] OS_Fingerprinting: 222 flujos (desde caché)


[8/24] Password: 91,167 flujos (desde caché)
[9/24] Port_Scanning: 10,934 flujos (desde caché)
[10/24] Ransomware: 1,431 flujos (desde caché)
[11/24] SQL_injection: 4,433 flujos (desde caché)
[12/24] Uploading: 7,635 flujos (desde caché)
[13/24] Vulnerability_scanner: 2,912 flujos (desde caché)
[14/24] XSS: 1,914 flujos (desde caché)
[15/24] Distance: 62,993 flujos (desde caché)
[16/24] Flame_Sensor: 62,959 flujos (desde caché)
[17/24] Heart_Rate: 10,587 flujos (desde caché)
[18/24] IR_Receiver: 63,158 flujos (desde caché)
[19/24] Modbus: 1,092 flujos (desde caché)
[20/24] Soil_Moisture: 59,306 flujos (desde caché)
[21/24] Sound_Sensor: 62,804 flujos (desde caché)
[22/24] Temperature_and_Humidity: 48,430 flujos (desde caché)
[23/24] Water_Level: 63,044 flujos (desde caché)
[24/24] phValue: 45,593 flujos (desde caché)
Características: 31


## Paso 3 · Etiquetado de los flujos de ataque
Las capturas de ataque también contienen tráfico legítimo de fondo (los sensores siguen publicando). Etiquetar todo el archivo como ataque introduciría ruido. Con `LABEL_MODE = "ip_no_vista"`, un flujo es de ataque si alguno de sus extremos es una IP **que nunca aparece en el tráfico normal** (el atacante o las IP falsificadas de las inundaciones); el resto se descarta como tráfico de fondo.

**Revisa la tabla:** si un ataque queda con `pct_ataque` cercano a 0, cambia a `LABEL_MODE = "manual"` con la IP del atacante que veas en `ips_iniciadoras_top`.

In [4]:
attack_flows, diag = label_attack_flows(flows, mode=LABEL_MODE, attacker_ips=ATTACKER_IPS)
diag.to_csv(RES_DIR / "diagnostico_etiquetado.csv", index=False, encoding="utf-8-sig")
diag

,ataque,flujos_archivo,flujos_ataque,flujos_descartados,pct_ataque,ips_iniciadoras_top
0,Backdoor,1471,921,550,62.6,"192.168.0.128 (877), 192.168.0.170 (44)"
1,DDoS_HTTP_Flood,12368,12339,29,99.8,"192.168.0.170 (12324), 192.168.0.128 (15)"
2,DDoS_ICMP_Flood,809594,755569,54025,93.3,"86.174.240.106 (2), 153.106.59.49 (2), 41.112...."
3,DDoS_TCP_SYN_Flood,572947,512429,60518,89.4,"192.168.0.128 (6), 32.167.23.1 (2), 40.221.49...."
4,DDoS_UDP_Flood,999929,926410,73519,92.6,"192.168.0.128 (107958), 168.101.21.145 (3), 14..."
5,MITM,244,2,242,0.8,192.168.0.128 (2)
6,OS_Fingerprinting,222,149,73,67.1,"192.168.0.170 (146), 192.168.0.128 (3)"
7,Password,91167,89747,1420,98.4,"192.168.0.170 (89658), 192.168.0.128 (89)"
8,Port_Scanning,10934,9987,947,91.3,192.168.0.170 (9987)
9,Ransomware,1431,650,781,45.4,"192.168.0.128 (642), 192.168.0.170 (8)"


## Paso 4 · Dataset final
Se eliminan vectores duplicados dentro de cada clase (para que filas idénticas no queden a la vez en entrenamiento y prueba), se limita cada clase/sensor a `MAX_PER_CLASS` flujos y se excluyen las clases con menos de 100 flujos etiquetados (en Edge-IIoTset, MITM es un ARP spoofing de capa 2 que casi no deja flujos IP).

In [5]:
data, report = build_dataset(flows, attack_flows, max_per_class=MAX_PER_CLASS)
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(OUT_FILE, index=False)
report.to_csv(RES_DIR / "tabla_dataset.csv", index=False, encoding="utf-8-sig")
print(f"Guardado {OUT_FILE.name}: {len(data):,} flujos, {data['label'].nunique()} clases")
report

Guardado flows_labeled.parquet: 176,118 flujos, 14 clases


,clase,dispositivo,flujos,sin_duplicados,usados,nota
0,Normal,Distance,62993,62869,15000,
1,Normal,Flame_Sensor,62959,62843,15000,
2,Normal,Heart_Rate,10587,10443,10443,
3,Normal,IR_Receiver,63158,63017,15000,
4,Normal,Modbus,1092,677,677,
5,Normal,Soil_Moisture,59306,58754,15000,
6,Normal,Sound_Sensor,62804,62696,15000,
7,Normal,Temperature_and_Humidity,48430,45331,15000,
8,Normal,Water_Level,63044,62932,15000,
9,Normal,phValue,45593,45218,15000,


In [6]:
# Descriptivos por categoría (útiles para la sección de Metodología del artículo)
desc = data.groupby("category").agg(flujos=("label", "size"),
                                    duracion_mediana_s=("duration", "median"),
                                    paquetes_medianos=("fwd_pkts", "median"),
                                    bytes_medianos=("fwd_bytes", "median"))
desc.to_csv(RES_DIR / "descriptivos_categoria.csv", encoding="utf-8-sig")
desc

,flujos,duracion_mediana_s,paquetes_medianos,bytes_medianos
category,,,,
DDoS,15135,63.912113,7.0,420.0
Inyeccion,10470,0.036639,6.0,491.0
Malware,16470,0.117121,6.0,406.0
Normal,131120,0.035248,8.0,384.0
Reconocimiento,2923,0.084726,12.0,1774.0
